# 05 — Symbolic Representation

This notebook assembles prototypes and temporal segments into symbolic sequences (Chapter 5, §5.4).

**Pipeline overview:**
1. **Config & Data Loading** — Load symbolization config, experiment DataFrame, clinical filtering
2. **Raw Chronograms** — Visualize frame-level embedding labels before segment aggregation
3. **Cluster DataFrame & Prevalence Filtering** — Build cluster statistics, filter low-prevalence prototypes
4. **Label & Distance Distributions** — Inspect symbol frequency and prototype distance distributions
5. **Cohort Symbolic Representation** — Visualize symbolic sequences across the full cohort
6. **Temporal Distribution Estimation** — KDE of prototype temporal profiles, sorted by mean timestamp
7. **Gromov-Wasserstein Temporal Distances** — Pairwise GW distance matrix between prototype KDEs
8. **Category-Level Labeling** — Map prototypes to expert-defined action categories (pyramid labels)
9. **Segment Length Analysis** — Duration distributions per pathology group, ECDF, LME statistical testing

## 1. Config & Data Loading

Load symbolization configuration, experiment DataFrames, and apply clinical filtering.

See: `smartflat.features.symbolization.utils_dataset.get_experiments_dataframe`,
     `smartflat.features.symbolization.utils.fix_clinical_diagnosis`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from collections import Counter
from IPython.display import display
from scipy import stats

from smartflat.configs.loader import import_config
from smartflat.constants import incomplete_clinical_administrations
from smartflat.constants_annotations_prototypes import (
    mapping_category_df,
    mapping_cluster_category,
)
from smartflat.features.symbolization.main import (
    build_clusterdf,
    filter_prototypes_per_prevalence,
)
from smartflat.features.symbolization.utils import (
    add_pyramid_labels,
    fix_clinical_diagnosis,
    get_prototypes,
    update_segmentation_from_embedding_labels,
)
from smartflat.features.symbolization.utils_dataset import get_experiments_dataframe
from smartflat.features.symbolization.co_clustering import compute_temporal_distance
from smartflat.features.symbolization.temporal_distributions_estimation import (
    estimate_kde_model,
    plot_clusters_kde_from_model,
    plot_gw_temporal_distance,
    prepare_row_timestamps_dataset,
    sort_kde_by_mean_timestamp,
)
from smartflat.features.symbolization.visualization import (
    plot_cluster_prevalence,
    plot_distances_distribution_subplots,
    plot_labels_counts_per_diagnosis,
    plot_labels_distributions_subplots,
    plot_symbols_frequency_subplots,
    plot_symbolization_variables,
    visualize_cohort_symbolic_representation,
)
from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils_visualization import plot_chronogames, plot_gram


In [ ]:
# --- Experiment parameters ---
annotator_id = 'samperochon'
round_number = 8

# SymbolicSourceInferenceCompleteGoldConfig: complete pipeline including all annotation rounds
# SymbolicSourceInferenceGoldConfig: final inference with definitive prototypes only
symbolization_config_name = 'SymbolicSourceInferenceCompleteGoldConfig'
config = import_config(symbolization_config_name)
clustering_config_name = config.clustering_config_name

input_space = 'G_space'

data_root = get_data_root()
print(f"Data root: {data_root}")
print(f"Clustering config: {clustering_config_name}")


In [ ]:
df = get_experiments_dataframe(
    experiment_config_name=symbolization_config_name,
    annotator_id=annotator_id,
    round_number=round_number,
    return_symbolization=True,
    return_data_only=True,
)
print(f"Loaded {len(df)} recordings")
display(df[['identifier', 'participant_id', 'task_name', 'modality', 'N', 'duration']].head(10))


In [ ]:
df = fix_clinical_diagnosis(df)
df = df[~df['participant_id'].isin(incomplete_clinical_administrations.keys())]
df.sort_values('task_number_int', ascending=True, inplace=True)
df = df[df['pathologie'].isin(['HEALTHY', 'RIL', 'TBI'])]
df.drop_duplicates(subset=['trigram'], keep='first', inplace=True)

print(f"After filtering: {len(df)} recordings")
display(df.groupby('pathologie')['group_folder'].value_counts().to_frame('count').T)


## 2. Raw Chronograms

Visualize frame-level embedding labels before segment-level aggregation. Each row is a participant's
video, color-coded by prototype assignment.

See: `smartflat.utils.utils_visualization.plot_chronogames`

In [ ]:
if df is not None:
    plot_chronogames(df, labels_col='raw_embedding_labels', n_subjects=8)
    plt.suptitle(f'Raw embedding labels (round {round_number})', y=1.02)
    plt.show()


## 3. Cluster DataFrame & Prevalence Filtering

Build a cluster-level DataFrame summarizing per-prototype statistics (subject count, occurrence count,
mean distance). Filter out low-prevalence prototypes appearing in fewer than 10 subjects or 10 times.

See: `smartflat.features.symbolization.main.build_clusterdf`,
     `smartflat.features.symbolization.main.filter_prototypes_per_prevalence`

In [ ]:
if df is not None:
    clusterdf = build_clusterdf(
        df,
        embeddings_label_col='opt_embedding_labels',
        segments_labels_col='opt_segments_labels',
        annotator_id=annotator_id,
        round_number=round_number,
        verbose=False,
    )
    print(f"Clusters: {len(clusterdf)}")
    display(clusterdf[['cluster_index', 'cluster_type', 'n_subjects', 'n_appearance',
                        'cluster_dist_mean']].head(15))


In [ ]:
if df is not None:
    excluded = filter_prototypes_per_prevalence(
        clusterdf, min_subjects=10, min_occurences=10, title='G-space', verbose=True,
    )
    print(f"Excluded {len(excluded)} low-prevalence clusters")

    plot_cluster_prevalence(
        df,
        segments_labels_col='opt_segments_labels',
        title=f'Prototype prevalence after filtering (round {round_number}, {input_space})',
        verbose=True,
    )
    plt.show()


## 4. Label & Distance Distributions

Inspect the distribution of symbol assignments across the cohort: how frequently each prototype
appears, and the distribution of distances between samples and their assigned prototype.

See: `smartflat.features.symbolization.visualization.plot_labels_distributions_subplots`,
     `smartflat.features.symbolization.visualization.plot_distances_distribution_subplots`

In [ ]:
if df is not None:
    plot_labels_distributions_subplots(
        df,
        clustering_config_names=[symbolization_config_name],
        labels_col='opt_embedding_labels',
        segments_labels_col='opt_segments_labels',
        clusterdf=clusterdf,
    )
    plt.show()


In [ ]:
if df is not None:
    plot_distances_distribution_subplots(
        df,
        clustering_config_names=[symbolization_config_name],
        title=f'Distance to assigned prototype ({input_space})',
    )
    plt.show()


In [ ]:
if df is not None:
    plot_symbols_frequency_subplots(
        df,
        clustering_config_names=[symbolization_config_name],
    )
    plt.show()


## 5. Cohort Symbolic Representation

Visualize symbolic sequences for the full cohort, sorted by a covariate (number of distinct symbols).
Each row is a participant; colors encode prototype identity.

See: `smartflat.features.symbolization.visualization.visualize_cohort_symbolic_representation`,
     `smartflat.features.symbolization.visualization.plot_symbolization_variables`

In [ ]:
if df is not None:
    visualize_cohort_symbolic_representation(
        df,
        covariate='n_cluster',
        labels_col='opt_embedding_labels',
    )
    plt.suptitle(f'Cohort symbolic representation ({input_space}, round {round_number})', y=1.02)
    plt.show()


In [ ]:
if df is not None:
    plot_symbolization_variables(df)
    plt.show()


## 6. Temporal Distribution Estimation

Estimate when each prototype appears during the video using kernel density estimation (KDE).
Timestamps are normalized to [0, 1] per participant, then aggregated with inverse-prevalence
weighting. KDEs are sorted by mean timestamp to reveal the temporal ordering of actions.

See: `smartflat.features.symbolization.temporal_distributions_estimation.estimate_kde_model`,
     `smartflat.features.symbolization.temporal_distributions_estimation.plot_clusters_kde_from_model`

In [ ]:
if df is not None:
    kde_models = estimate_kde_model(
        config_name=symbolization_config_name,
        df=df,
        annotator_id=annotator_id,
        round_number=round_number,
        input_space=input_space,
        verbose=True,
    )
    kde_model = kde_models['foreground']
    print(f"Fitted KDE for {len(kde_model)} foreground prototypes")


In [ ]:
if df is not None and kde_model is not None:
    cluster_timestamps, timestamps_weights, _, cluster_prevalence = prepare_row_timestamps_dataset(df)
    kde_model_sorted = sort_kde_by_mean_timestamp(kde_model)

    plot_clusters_kde_from_model(
        df,
        kde_model_sorted,
        cluster_timestamps,
        cluster_prevalence,
        n_clusters=min(len(kde_model_sorted), 48),
        title=f'Temporal KDE per prototype ({input_space}, round {round_number})',
        n_cols=6,
    )


## 7. Gromov-Wasserstein Temporal Distances

Compute pairwise Gromov-Wasserstein distances between prototype temporal distributions.
This captures structural differences in when prototypes appear during the task,
beyond simple mean-shift comparisons.

See: `smartflat.features.symbolization.co_clustering.compute_temporal_distance`,
     `smartflat.features.symbolization.temporal_distributions_estimation.plot_gw_temporal_distance`

In [ ]:
if df is not None:
    D_temporal = compute_temporal_distance(
        kde_models=kde_models['foreground'],
        df=df,
        annotator_id=annotator_id,
        round_number=round_number,
        config_name=symbolization_config_name,
        input_space=input_space,
        overwrite_gw_distances=False,
        ordered_index=True,
        verbose=True,
    )
    print(f"Temporal distance matrix shape: {D_temporal.shape}")
    print(f"Range: [{D_temporal[D_temporal > 0].min():.4f}, {D_temporal.max():.4f}]")


In [ ]:
if df is not None and D_temporal is not None:
    plot_gw_temporal_distance(D_temporal, title=f'GW Temporal Distance ({input_space})')
    plt.show()


## 8. Category-Level Labeling

Map individual prototypes to expert-defined action categories (e.g., "use mixer", "break eggs",
"manipulate oven") using the pyramid label hierarchy. This produces coarser symbolic sequences
where segments are labeled by action category rather than prototype index.

See: `smartflat.features.symbolization.utils.add_pyramid_labels`,
     `smartflat.constants_annotations_prototypes.mapping_cluster_category`

In [ ]:
if df is not None:
    df = add_pyramid_labels(df, symbolization_config_name, annotator_id, round_number)

    # Map prototype indices to action category names
    df['category_labels'] = df['pyr_filtered_raw_embedding_labels'].apply(
        lambda x: [mapping_cluster_category[i][0] if i in mapping_cluster_category else '-1' for i in x]
    )

    n_categories = len(set(c for cats in df['category_labels'] for c in cats if c != '-1'))
    print(f"Pyramid labels added — {n_categories} action categories")
    display(pd.Series(Counter(c for cats in df['category_labels'] for c in cats if c != '-1')).sort_values(ascending=False).head(15).to_frame('count'))


In [ ]:
if df is not None:
    col_names = ['cat_segm_embedding_labels', 'cat_segments_labels',
                 'cat_cpts', 'cat_n_cpts', 'cat_segments_has_separations', 'cat_n_segmented',
                 'cat_segments_length', 'cat_n_embed_changes', 'cat_percent_embed_changes',
                 'cat_sum_cpts_withdrawn', 'cat_percent_cpts_withdrawn']

    df[col_names] = df.apply(
        update_segmentation_from_embedding_labels, axis=1,
        temporal_segmentation_col='raw_cpts',
        embedding_labels_col='category_labels',
        combine_func_name='majority_voting_inner',
        filter_noise_labels=False,
        verbose=False,
    )

    plot_chronogames(df, labels_col='cat_segm_embedding_labels', n_subjects=8,
                     time_calibration='embeddings')
    plt.suptitle(f'Category-level symbolic sequences (round {round_number})', y=1.02)
    plt.show()


## 9. Segment Length Analysis

Analyze the distribution of symbolic segment durations across pathology groups.
Segments are exploded from the per-participant arrays, converted to seconds, and
compared between HEALTHY and patient groups using empirical CDFs and
linear mixed-effects (LME) regression.

**LME model:** log(segment_duration) ~ C(pathologie) + (1 | participant_id)

In [ ]:
if df is not None:
    # Explode segments into one row per segment (delta_t=8 frames, fps=25)
    df['raw_segments_length'] = df['raw_cpts'].apply(np.ediff1d) * 8 / 25

    long_df = df[['participant_id', 'group_folder', 'pathologie', 'raw_segments_length']].explode('raw_segments_length')
    long_df['raw_segments_length'] = long_df['raw_segments_length'].astype(float)
    long_df = long_df[long_df['raw_segments_length'] > 0]
    long_df['log_len'] = np.log(long_df['raw_segments_length'])

    print(f"Total segments: {len(long_df)}")
    display(long_df.groupby('pathologie').agg(
        n_participants=('participant_id', 'nunique'),
        n_segments=('log_len', 'count'),
        median_s=('raw_segments_length', 'median'),
        mean_s=('raw_segments_length', 'mean'),
    ).round(2))


In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(14, 5))

    # Individual participants in grey
    for pid in df['participant_id'].unique():
        segs = long_df.loc[long_df['participant_id'] == pid, 'raw_segments_length']
        sns.ecdfplot(segs, color='gray', alpha=0.3, linewidth=0.5, ax=ax)

    # Per-pathology group ECDFs
    colors = {'HEALTHY': 'tab:blue', 'RIL': 'tab:red', 'TBI': 'tab:orange'}
    for patho, color in colors.items():
        sub = long_df[long_df['pathologie'] == patho]
        if len(sub) > 0:
            sns.ecdfplot(sub['raw_segments_length'], label=patho, color=color, linewidth=2.5, ax=ax)

    ax.set_xscale('log')
    ax.set(xlabel='Segment duration (seconds)', ylabel='ECDF',
           title='Empirical CDF of segment durations per pathology group')
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()


In [ ]:
if df is not None:
    # Null model: no group fixed effect
    null = smf.mixedlm('log_len ~ 1', long_df, groups=long_df['participant_id'])
    res_null = null.fit(reml=False)

    # Alternative model: pathologie fixed effect
    alt = smf.mixedlm('log_len ~ C(pathologie)', long_df, groups=long_df['participant_id'])
    res_alt = alt.fit(reml=False)

    # Likelihood ratio test
    llr = 2 * (res_alt.llf - res_null.llf)
    df_diff = res_alt.df_modelwc - res_null.df_modelwc
    pval = stats.chi2.sf(llr, df_diff)

    print(f"Likelihood ratio test: LR={llr:.3f}, df={df_diff}, p={pval:.4e}")
    print(f"\nFixed effects (log scale):")
    for param, coef in res_alt.fe_params.items():
        print(f"  {param}: coef={coef:.4f}, multiplicative effect={np.exp(coef)-1:.3f}")

    # ICC
    var_u = res_alt.cov_re.iloc[0, 0]
    var_eps = res_alt.scale
    icc = var_u / (var_u + var_eps)
    print(f"\nICC (participant): {icc:.3f}")
    print(res_alt.summary())


In [ ]:
if df is not None:
    plot_labels_counts_per_diagnosis(df)
    plt.suptitle(f'Symbol counts per diagnosis ({input_space}, round {round_number})', y=1.02)
    plt.show()
